# WSmart+ Route — Daily Output Analysis

This notebook analyses the **day-by-day** results stored in the JSON log files produced by simulation runs (`assets/output/`).  
Unlike `output.ipynb` (which focuses on end-of-simulation summaries), this notebook
decomposes each 30-day run into its daily time-series to reveal:

- When collections happen and how frequently
- How metrics evolve across the simulation period
- Which days are problematic (overflows, low efficiency)
- How policies differ at the day level, not just overall

**Log format per policy** (`log_<slug>_1N.json`):
```
{ samples: {0: {summary KPIs}},
  daily:   {0: {overflows:[…], kg:[…], km:[…], kg/km:[…], … tour:[…]  }},
  mean: {…}, std: {…} }
```

In [ ]:
from tutorials.notebook_setup import setup_google_colab, setup_home_directory

NOTEBOOK_NAME = 'daily_output'
home_dir = setup_home_directory(NOTEBOOK_NAME)
IN_COLAB, gdrive, gfiles = setup_google_colab(NOTEBOOK_NAME)

In [ ]:
import glob, json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 130, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

DAILY_METRICS = ['overflows', 'kg', 'ncol', 'kg_lost', 'km', 'kg/km', 'reward', 'profit', 'time']
METRIC_LABELS = {
    'overflows': 'Overflows (count)',
    'kg':        'Waste Collected (kg)',
    'ncol':      'Collections (count)',
    'kg_lost':   'Waste Lost / Overflowed (kg)',
    'km':        'Distance Driven (km)',
    'kg/km':     'Efficiency (kg/km)',
    'reward':    'Reward',
    'profit':    'Profit',
    'time':      'Solve Time (s)',
}
COLORS = ['steelblue','darkorange','mediumseagreen','tomato','mediumpurple',
          'saddlebrown','deeppink','teal']

---
## 1 · Configuration

In [ ]:
# ── Set these to match your simulation run ────────────────────────────────────
N_DAYS      = 30          # simulation length
N_BINS      = 100         # number of bins
AREA        = 'Rio Maior' # area name (as stored in directory)
WASTE_TYPE  = 'plastic'
DIST        = 'gamma3'    # data distribution ('gamma3' | 'emp')
RUN_NAME    = 'la_ftsp'   # subdirectory run name

# ── Derived directory path (matches assets/output structure) ──────────────────
area_slug = re.sub(r'[^a-zA-Z0-9]', '', AREA.lower())
LOG_DIR = os.path.join(
    home_dir, 'assets', 'output',
    f'{N_DAYS}days',
    f'{area_slug}{N_BINS}_{WASTE_TYPE}',
    DIST,
    RUN_NAME,
)
print('Log directory:', LOG_DIR)
print('Exists:', os.path.isdir(LOG_DIR))
json_files = sorted(glob.glob(os.path.join(LOG_DIR, 'log_*.json')))
print(f'Found {len(json_files)} log file(s)')
for f in json_files:
    print(' ', os.path.basename(f))

---
## 2 · Load Daily Results

In [ ]:
def _load_daily_df(json_files, n_days, sample_key='0'):
    """Parse all log files into a flat daily DataFrame.

    Columns: policy, day (0-indexed), overflows, kg, ncol, kg_lost,
             km, kg/km, reward, profit, time, has_collection, n_mandatory.
    """
    records = []
    policy_meta = {}
    for path in json_files:
        basename = os.path.basename(path)
        # slug = everything between 'log_' and '_1N.json' (or last '_<n>N.json')
        slug = re.sub(r'^log_', '', re.sub(r'_\d+N\.json$', '', basename))
        with open(path) as fh:
            data = json.load(fh)

        daily = data.get('daily', {}).get(sample_key, {})
        summary = data.get('samples', {}).get(sample_key, data.get('mean', {}))
        policy_meta[slug] = {m: summary.get(m, None) for m in DAILY_METRICS}

        n_actual = len(daily.get('overflows', []))
        for day in range(min(n_days, n_actual)):
            rec = {'policy': slug, 'day': day}
            for m in DAILY_METRICS:
                vals = daily.get(m, [])
                rec[m] = vals[day] if day < len(vals) else None
            tour     = daily.get('tour', [[]] * (day+1))
            mand     = daily.get('mandatory_nodes', [[]] * (day+1))
            rec['has_collection'] = len(tour[day]) > 1 if day < len(tour) else False
            rec['n_mandatory']    = len(mand[day]) if day < len(mand) else 0
            records.append(rec)

    df = pd.DataFrame(records)
    df.rename(columns={'kg/km': 'kg_per_km'}, inplace=True)
    return df, policy_meta

df_daily, policy_meta = _load_daily_df(json_files, N_DAYS)
policies = sorted(df_daily['policy'].unique())
print(f'Loaded {len(policies)} policies × {N_DAYS} days = {len(df_daily)} rows')
print('Policies:', policies)
df_daily.head(10)

In [ ]:
# ── Summary: which days had collections? ─────────────────────────────────────
collection_days = (
    df_daily.groupby(['policy','day'])['has_collection']
    .first()
    .unstack('policy')
    .astype(int)
)
print(f'Collection calendar ({N_DAYS} days × {len(policies)} policies):')
display(collection_days)

---
## 3 · Day-by-Day Metric Line Charts

In [ ]:
PLOT_METRICS = ['overflows', 'kg', 'km', 'kg_per_km']  # choose up to 4

n_m  = len(PLOT_METRICS)
fig, axes = plt.subplots(n_m, 1, figsize=(14, 3.5 * n_m), sharex=True)
axes = np.atleast_1d(axes)
days_x = np.arange(N_DAYS)

for ax, metric in zip(axes, PLOT_METRICS):
    col_label = metric.replace('kg_per_km', 'kg/km')
    for pol, col in zip(policies, COLORS):
        sub = df_daily[df_daily['policy'] == pol].sort_values('day')
        vals = sub[metric if metric != 'kg_per_km' else 'kg_per_km'].values
        ax.plot(days_x, vals, color=col, lw=1.8, label=pol, marker='o', markersize=3)
    ax.set_ylabel(METRIC_LABELS.get(col_label, metric), fontsize=9)
    ax.axhline(0, color='black', lw=0.5, alpha=0.3)

axes[0].set_title(f'Day-by-Day Metrics — {N_BINS} bins / {DIST} / {RUN_NAME}', fontsize=12)
axes[-1].set_xlabel('Simulation Day')
axes[0].legend(fontsize=8, loc='upper right', ncol=min(4, len(policies)), framealpha=0.9)
plt.tight_layout(); plt.show()

---
## 4 · Cumulative Metrics Over Days

In [ ]:
CUM_METRICS = ['overflows', 'kg', 'km']   # metrics to cumulate

n_c  = len(CUM_METRICS)
fig, axes = plt.subplots(1, n_c, figsize=(6 * n_c, 4))
axes = np.atleast_1d(axes)

for ax, metric in zip(axes, CUM_METRICS):
    for pol, col in zip(policies, COLORS):
        sub  = df_daily[df_daily['policy'] == pol].sort_values('day')
        vals = sub[metric].fillna(0).values
        ax.plot(days_x, np.cumsum(vals), color=col, lw=2, label=pol)
    ax.set_title(f'Cumulative {metric}'); ax.set_xlabel('Day')
    ax.set_ylabel(f'Cumulative {METRIC_LABELS.get(metric, metric)}', fontsize=9)

axes[0].legend(fontsize=8, ncol=2, framealpha=0.9)
plt.suptitle(f'Cumulative Metrics — {DIST} / {RUN_NAME}', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

---
## 5 · Distribution of Daily KPIs (Box Plots per Policy)

In [ ]:
BOX_METRICS = ['overflows', 'kg', 'km', 'kg_per_km']

n_bm = len(BOX_METRICS)
fig, axes = plt.subplots(1, n_bm, figsize=(5 * n_bm, 5))
axes = np.atleast_1d(axes)

for ax, metric in zip(axes, BOX_METRICS):
    data_per_pol = []
    for pol in policies:
        sub = df_daily[df_daily['policy'] == pol].sort_values('day')
        col = metric if metric != 'kg_per_km' else 'kg_per_km'
        # include only collection days to avoid zero-inflation
        vals = sub.loc[sub['has_collection'], col].dropna().values
        data_per_pol.append(vals if len(vals) > 0 else np.array([0.0]))

    bp = ax.boxplot(data_per_pol, patch_artist=True,
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    for patch, col in zip(bp['boxes'], COLORS):
        patch.set_facecolor(col); patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(policies)+1))
    ax.set_xticklabels([p.split('_')[0] for p in policies],
                       rotation=30, ha='right', fontsize=8)
    ax.set_title(METRIC_LABELS.get(metric.replace('kg_per_km','kg/km'), metric))
    ax.set_ylabel('Value (collection days only)', fontsize=8)

plt.suptitle(f'Daily KPI Distributions — {DIST} / {RUN_NAME}', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

---
## 6 · Policy × Day Heatmaps

In [ ]:
HEAT_METRICS = ['overflows', 'kg', 'km', 'kg_per_km']

for metric in HEAT_METRICS:
    col = metric if metric != 'kg_per_km' else 'kg_per_km'
    mat = np.zeros((len(policies), N_DAYS))
    for pi, pol in enumerate(policies):
        sub = df_daily[df_daily['policy'] == pol].sort_values('day')
        for _, row in sub.iterrows():
            d = int(row['day'])
            if d < N_DAYS:
                v = row[col]
                mat[pi, d] = 0.0 if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)

    fig, ax = plt.subplots(figsize=(14, max(3, len(policies) * 0.55)))
    vmax = float(np.percentile(mat[mat > 0], 98)) if (mat > 0).any() else 1.0
    im = ax.imshow(mat, aspect='auto', cmap='YlOrRd', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label=METRIC_LABELS.get(metric.replace('kg_per_km','kg/km'), metric),
                 shrink=0.8)
    ax.set_yticks(range(len(policies)))
    ax.set_yticklabels(policies, fontsize=8)
    ax.set_xlabel('Simulation Day')
    ax.set_title(f'{METRIC_LABELS.get(metric.replace("kg_per_km","kg/km"), metric)} '
                 f'— Policy × Day  ({DIST} / {RUN_NAME})')
    plt.tight_layout(); plt.show()

---
## 7 · Collection Day Analysis

In [ ]:
# ── Collection frequency per policy ──────────────────────────────────────────
col_freq = (
    df_daily.groupby('policy')['has_collection'].mean() * 100
).rename('Collection Rate (%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

bars = axes[0].bar(range(len(policies)), col_freq[policies].values,
                   color=COLORS[:len(policies)], alpha=0.8, edgecolor='white')
axes[0].set_xticks(range(len(policies)))
axes[0].set_xticklabels([p.split('_')[0] for p in policies], rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('Collection Rate (%)')
axes[0].set_title('Collection Frequency per Policy')
for bar, v in zip(bars, col_freq[policies].values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

# ── Collection day calendar (binary heatmap) ──────────────────────────────────
cal_mat = np.zeros((len(policies), N_DAYS))
for pi, pol in enumerate(policies):
    sub = df_daily[df_daily['policy'] == pol].sort_values('day')
    for _, row in sub.iterrows():
        d = int(row['day'])
        if d < N_DAYS:
            cal_mat[pi, d] = 1 if row['has_collection'] else 0

im = axes[1].imshow(cal_mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_yticks(range(len(policies)))
axes[1].set_yticklabels(policies, fontsize=8)
axes[1].set_xlabel('Simulation Day')
axes[1].set_title('Collection Calendar (green = collection day)')
plt.colorbar(im, ax=axes[1], ticks=[0,1], label='Collection', shrink=0.8)

plt.suptitle(f'Collection Day Analysis — {N_BINS} bins / {DIST} / {RUN_NAME}', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

# Summary table
n_collections = df_daily.groupby('policy')['has_collection'].sum().astype(int)
kg_per_col    = df_daily[df_daily['has_collection']].groupby('policy')['kg'].mean().round(1)
km_per_col    = df_daily[df_daily['has_collection']].groupby('policy')['km'].mean().round(1)
summary_col   = pd.DataFrame({'# Collections': n_collections,
                               'Mean kg/col': kg_per_col,
                               'Mean km/col': km_per_col,
                               'Collection rate (%)': col_freq.round(1)})
display(summary_col)

---
## 8 · Multi-Run Comparison (Across run_names)

In [ ]:
# ── Compare la_ftsp / lm_ftsp / sl_ftsp for one policy ────────────────────────
COMPARE_POLICY = policies[0]   # pick a policy present in all run_names
RUN_NAMES_CMP  = ['la_ftsp', 'lm_ftsp', 'sl_ftsp']
METRIC_CMP     = 'kg_per_km'

run_dfs = {}
for rn in RUN_NAMES_CMP:
    rn_dir = os.path.join(
        home_dir, 'assets', 'output', f'{N_DAYS}days',
        f'{area_slug}{N_BINS}_{WASTE_TYPE}', DIST, rn,
    )
    if not os.path.isdir(rn_dir):
        print(f'  {rn}: directory not found'); continue
    files = sorted(glob.glob(os.path.join(rn_dir, 'log_*.json')))
    if not files:
        print(f'  {rn}: no log files'); continue
    df_r, _ = _load_daily_df(files, N_DAYS)
    run_dfs[rn] = df_r
    print(f'  {rn}: {df_r["policy"].nunique()} policies')

if len(run_dfs) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Line chart: metric over days for one policy across all run_names
    for rn, col in zip(run_dfs, COLORS):
        df_r = run_dfs[rn]
        # find a matching policy (partial name match)
        match = [p for p in df_r['policy'].unique()
                 if COMPARE_POLICY.split('_')[0] in p]
        if not match:
            continue
        pol = match[0]
        sub = df_r[df_r['policy'] == pol].sort_values('day')
        axes[0].plot(sub['day'].values, sub[METRIC_CMP].values,
                     color=col, lw=2, label=rn, marker='o', markersize=3)
    axes[0].set_title(f'{METRIC_CMP} over days — {COMPARE_POLICY.split("_")[0]} across runs')
    axes[0].set_xlabel('Day'); axes[0].set_ylabel(METRIC_LABELS.get(METRIC_CMP, METRIC_CMP))
    axes[0].legend()

    # Bar chart: mean metric across all policies per run_name
    rn_labels, rn_means = [], []
    for rn, df_r in run_dfs.items():
        m = df_r[df_r['has_collection']][METRIC_CMP].mean()
        rn_labels.append(rn); rn_means.append(m)
    axes[1].bar(rn_labels, rn_means, color=COLORS[:len(rn_labels)], alpha=0.8, edgecolor='white')
    axes[1].set_title(f'Mean {METRIC_CMP} (collection days only) — All Policies')
    axes[1].set_ylabel(METRIC_LABELS.get(METRIC_CMP, METRIC_CMP))

    plt.suptitle(f'Run-Name Comparison — {N_BINS} bins / {DIST}', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

---
## 9 · Interactive Plotly Charts

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# ── Interactive line chart: all policies, all days ────────────────────────────
IPLOT_METRIC = 'kg_per_km'

df_plot = df_daily[['policy','day', IPLOT_METRIC]].copy()
df_plot.rename(columns={IPLOT_METRIC: 'value'}, inplace=True)
df_plot['metric'] = METRIC_LABELS.get(IPLOT_METRIC.replace('kg_per_km','kg/km'), IPLOT_METRIC)

fig = px.line(
    df_plot, x='day', y='value', color='policy',
    template='plotly_dark',
    title=f'Day-by-Day {METRIC_LABELS.get(IPLOT_METRIC.replace("kg_per_km","kg/km"), IPLOT_METRIC)}'
          f' — {N_BINS} bins / {DIST} / {RUN_NAME}',
    labels={'day': 'Simulation Day', 'value': METRIC_LABELS.get(IPLOT_METRIC, IPLOT_METRIC),
            'policy': 'Policy'},
    markers=True,
)
fig.update_traces(mode='lines+markers', marker_size=5)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40), hovermode='x unified')
fig.show()

In [ ]:
# ── Animated scatter: kg vs km per day ───────────────────────────────────────
df_anim = df_daily[df_daily['has_collection']].copy()
df_anim['kg_per_km'] = df_anim['kg_per_km'].fillna(0)

fig2 = px.scatter(
    df_anim, x='km', y='kg', size='kg_per_km',
    color='policy', animation_frame='day',
    template='plotly_dark',
    title=f'Animated Daily KG vs KM (collection days only) — {DIST} / {RUN_NAME}',
    labels={'km': 'Distance (km)', 'kg': 'Waste collected (kg)',
            'kg_per_km': 'Efficiency (kg/km)', 'policy': 'Policy'},
    size_max=30,
    hover_data=['policy','day','overflows','kg_per_km'],
)
fig2.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig2.show()

In [ ]:
# ── Multi-metric comparison table (daily means per policy) ───────────────────
summary_rows = []
for pol in policies:
    sub = df_daily[df_daily['policy'] == pol]
    col_days = sub[sub['has_collection']]
    row = {'Policy': pol, '# Col days': int(sub['has_collection'].sum())}
    for m in ['overflows', 'kg', 'km', 'kg_per_km']:
        vals = col_days[m].dropna().values
        row[f'Mean {m}'] = round(float(vals.mean()), 3) if len(vals) > 0 else None
    summary_rows.append(row)
display(pd.DataFrame(summary_rows).set_index('Policy'))